In [ ]:
# Self trained
# ============================================================
# INSTALL
# ============================================================
!pip install pandas imapclient pyzmail36 nest_asyncio openai --quiet

# ============================================================
# IMPORTS
# ============================================================
import imapclient
import pyzmail
import pandas as pd
import nest_asyncio
import json
import os
from datetime import datetime
from getpass import getpass
from openai import OpenAI

nest_asyncio.apply()

# ============================================================
# CONFIG
# ============================================================
IMAP_SERVER = "imap.gmail.com"

EMAIL = input("Enter your email: ")
PASSWORD = getpass("Enter Gmail App Password: ")
GROQ_API_KEY = input("Enter Groq API Key: ")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

FOLDERS = [
    "INBOX",
    "[Gmail]/Sent Mail",
    "[Gmail]/Drafts",
    "[Gmail]/Important",
    "[Gmail]/Starred"
]

# ============================================================
# LOAD / SAVE LEARNED KEYWORDS
# ============================================================
LEARN_FILE = "learned_keywords.json"

if os.path.exists(LEARN_FILE):
    with open(LEARN_FILE, "r") as f:
        learned_keywords = json.load(f)
else:
    learned_keywords = {}

def save_learnings():
    with open(LEARN_FILE, "w") as f:
        json.dump(learned_keywords, f, indent=2)

# ============================================================
# MASSIVE KEYWORD ENGINE
# ============================================================
def get_base_keywords():
    return {

        "Offer": [
            "offer letter","congratulations","selected","employment offer",
            "welcome aboard","joining","ctc","salary","compensation",
            "offer rollout","final selection","offer acceptance"
        ],

        "Interview": [
            "interview","round","technical","hr round","final round",
            "assessment","test","assignment","panel","discussion",
            "screening","shortlisted","interview invite"
        ],

        "New Opportunity": [
            "job opportunity","hiring","opening","vacancy","role opportunity",
            "we are hiring","position available","jd attached",
            "job description","career opportunity","looking for candidates"
        ],

        "Rejected": [
            "regret","unfortunately","not selected","not shortlisted",
            "rejected","not a fit","position filled","application unsuccessful"
        ],

        "Urgent": [
            "urgent","immediate","asap","priority","today","tomorrow",
            "deadline","action required","important","quick response"
        ],

        "Alerts": [
            "alert","notification","reminder","update","system alert",
            "security alert","account alert","important update"
        ],

        "Spam": [
            "sale","discount","offer deal","win","lottery","free",
            "click here","subscribe","buy now","limited offer"
        ]
    }

# ============================================================
# RULE + LEARNING CLASSIFIER
# ============================================================
def rule_based_classification(text):
    text = text.lower()
    keywords = get_base_keywords()

    # Merge learned keywords
    for cat in learned_keywords:
        keywords.setdefault(cat, []).extend(learned_keywords[cat])

    scores = {cat: 0 for cat in keywords}

    for category, words in keywords.items():
        for word in words:
            if word in text:
                scores[category] += 1

    best_category = max(scores, key=scores.get)

    if scores[best_category] == 0:
        return "Unknown"

    return best_category

# ============================================================
# SELF LEARNING FUNCTION
# ============================================================
def learn_from_email(text, category):
    words = text.lower().split()

    # pick meaningful words
    filtered = [w for w in words if len(w) > 6]

    if category not in learned_keywords:
        learned_keywords[category] = []

    for w in filtered[:5]:
        if w not in learned_keywords[category]:
            learned_keywords[category].append(w)

    save_learnings()

# ============================================================
# LLM CLASSIFIER (PRIMARY)
# ============================================================
def classify_email(subject, body):
    text = (subject + " " + body)[:1500]

    prompt = f"""
    Classify this email into one:
    New Opportunity, Interview, Spam, Unknown, Offer, Rejected, Urgent, Alerts

    Extract company, role, round if present.

    Email:
    {text}

    Return JSON:
    {{
        "category": "",
        "company": "",
        "role": "",
        "round": "",
        "confidence": 0.0
    }}
    """

    try:
        res = client.chat.completions.create(
            model="llama3-70b-8192",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        output = res.choices[0].message.content.strip()
        output = output.replace("```", "").replace("json", "").strip()

        result = json.loads(output)

        # LEARN FROM HIGH CONFIDENCE
        if result["confidence"] > 0.8:
            learn_from_email(text, result["category"])

        return result

    except:
        category = rule_based_classification(text)

        learn_from_email(text, category)

        return {
            "category": category,
            "company": "Unknown",
            "role": "Unknown",
            "round": "N/A",
            "confidence": 0.6
        }

# ============================================================
# FETCH EMAILS
# ============================================================
def fetch_all_emails(limit=20):
    mail = imapclient.IMAPClient(IMAP_SERVER, ssl=True)
    mail.login(EMAIL, PASSWORD)

    all_emails = []

    for folder in FOLDERS:
        try:
            mail.select_folder(folder)
            messages = mail.search(["ALL"])
            messages = messages[-limit:]

            for uid in messages:
                raw = mail.fetch(uid, ["BODY[]"])
                msg = pyzmail.PyzMessage.factory(raw[uid][b"BODY[]"])

                subject = msg.get_subject()

                if msg.text_part:
                    body = msg.text_part.get_payload().decode(msg.text_part.charset)
                elif msg.html_part:
                    body = msg.html_part.get_payload().decode(msg.html_part.charset)
                else:
                    body = ""

                all_emails.append({
                    "folder": folder,
                    "subject": subject,
                    "body": body
                })

        except:
            continue

    mail.logout()
    return all_emails

# ============================================================
# MAIN PIPELINE
# ============================================================
def run_agent(limit=20):
    emails = fetch_all_emails(limit)
    records = []

    print("\n--- Processing Emails ---")

    for mail in emails:
        print(f"\n[{mail['folder']}] {mail['subject']}")

        result = classify_email(mail["subject"], mail["body"])

        record = {
            "timestamp": datetime.now(),
            "folder": mail["folder"],
            "company": result.get("company", "Unknown"),
            "role": result.get("role", "Unknown"),
            "category": result.get("category", "Unknown"),
            "round": result.get("round", "N/A"),
            "confidence": result.get("confidence", 0.5),
            "subject": mail["subject"]
        }

        records.append(record)

        print(f"→ {record['category']} | {record['company']}")

    df = pd.DataFrame(records)

    tracker = df.groupby(["company", "role"]).last().reset_index()

    df.to_csv("email_log.csv", index=False)
    tracker.to_csv("job_tracker.csv", index=False)

    print("\nSaved: email_log.csv & job_tracker.csv")

    print("\n--- STATUS COUNTS ---")
    print(tracker["category"].value_counts())

    return tracker

# ============================================================
# RUN
# ============================================================
tracker_df = run_agent(limit=20)

tracker_df

